# 从 4 张 collapsed 年度表构建 `full_data.dta`（30 列）

清洗逻辑照搬 `IO_Table/io_repro`（`02_cleaning_pipeline.do` + `pre_process.ipynb`），**唯一区别是全程保留 `year`**。IO 表口径把 2017+2018 合并成一张截面，本项目要年度面板。

## 数据流

```
Data/  buyer_17_collapsed / buyer_18_collapsed         (只读，不修改)
       seller_17_collapsed / seller_18_collapsed
       bianma_all.dta (19 位编码表)
 │ [01_cleaning.do]  改列名 + 加 year + input_output → append → v=正+负
 │                   → 产品码补 19 位 + 数值过滤 → 并编码表
 │                   → collapse by(firm product io **year**) → drop v<=0
 │                   → 截 15 位 + is_output + 只留有产出企业
 ▼ lenth15_year.dta
 │ [Step1] 15→9 位码标准化（层级码处理）+ firm 交集
 ▼ lenth9_year.dta
 │ [Step2] firm×product×year 聚合；外包额 = min(投入, 产出)
 ▼ firm_product_year_level.dta            (7 列)
 │ [Step3] 产品级特征聚合
 ▼ product_characteristics.dta            (11 列, 约 2,778 产品)
 │ [Step4] firm×year 汇总：外包强度、中介/外包标记
 │ [Step5] 主产品 = production_value 最大
 │ [Step6] 合并 similarity + 产品特征（_p 后缀）
 ▼ full_data.dta                          (30 列)
 │ [Step7] 与现有 full_data 对比验证
 │ [Step8] 描述统计
```

## 与 IO 表口径的两处**刻意不同**

| io_repro 的步骤 | 本 pipeline | 原因 |
|---|---|---|
| `lenth9_clean`：对角线对冲（同企业同产品买卖轧差） | **不做** | 外包的定义就是"同企业同产品既买又卖"，对冲会把要研究的信息抹掉 |
| `lenth9_domin`：主导产品处理（占比 >0.99 砍零头） | **不做** | IO 表估系数专用的调整，与本项目无关 |

## 关键口径

- **外包额** = `min(投入额, 产出额)`（逐 firm×product×year）——只算"买来又卖掉"的部分。
- **自产额** `production_value` = 产出额 − 外包额。
- **外包强度** = `Σ外包额 / Σ产出额`（firm×year）。
- **中介** `is_intermediary` = 强度 > 0.90；**外包企业** `is_outsourcing` = 强度 ≥ 0.01。
- **主产品** = firm×year 内 **`production_value` 最大**（并列取 `product_id` 最小）。

> ⚠️ **内存**：`lenth15_year` 量级在 5–7 亿行，Step 1 用分块读入 + 块内聚合。需在 VM（大内存）运行。

In [2]:
import pandas as pd
import numpy as np
import gc, os, subprocess
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# ==== 路径：原始数据只读，所有产物写 OUT ====
DATA = Path(r'G:\Kuangyu_Temp\Data')                          # 原始数据（只读）
OUT  = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1_data')     # 所有产物
CODE = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1')          # 代码（git）
# 本地：
# DATA = Path(r'C:\Users\HKUBS\Documents\aproject\Data')
# OUT  = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1_data')
# CODE = Path(r'C:\Users\HKUBS\Documents\aproject\Outsourcing\Empirical1')

SIM     = DATA / 'full_product_similarity.dta'   # 产品对相似度
BIANMA9 = DATA / 'bianma.dta'                    # 9 位编码表（2,778），用于核对

LENTH15      = OUT / 'lenth15_year.dta'          # 01_cleaning.do 的产物
OUT_LENTH9   = OUT / 'lenth9_year.dta'
OUT_FPY      = OUT / 'firm_product_year_level.dta'
OUT_PCHARS   = OUT / 'product_characteristics.dta'
OUT_FULLDATA = OUT / 'full_data.dta'
OLD_FULLDATA = DATA / 'full_data.dta'            # 现有底表（Step 7 对比用；没有就跳过）

OUT.mkdir(exist_ok=True)
os.chdir(OUT)
print('DATA:', DATA)
print('OUT :', OUT)

DATA: G:\Kuangyu_Temp\Data
OUT : G:\Kuangyu_Temp\Outsource\Empirical1_data


## Step 0　跑 `01_cleaning.do`（Stata）→ `lenth15_year.dta`

超大文件的 append / 产品码清洗 / collapse 交给 Stata（精确、无分块问题）。

**已经跑过就跳过这一格**，直接从 Step 1 开始。

In [ ]:
stata_exe = r"C:/Program Files/Stata17/StataMP-64.exe"
do_file   = str(CODE / 'pipeline' / '01_cleaning.do')

print('running 01_cleaning.do ...')
proc = subprocess.run([stata_exe, "/e", "do", do_file],
                      capture_output=True, text=True, errors="ignore")
print('Stata return code:', proc.returncode, '(0 = 正常)')
print('lenth15_year 大小: {:.2f} GB'.format(LENTH15.stat().st_size / 1e9))

running 01_cleaning.do ...


## Step 1　15 位码 → 9 位码标准化 + firm 交集

商品编码分层（1/3/5/7/9 位为层级节点，后面补 0）。要统一到 9 位，但有些产品实际只细到 5 或 7 位：

1. 剔除**高层聚合码**（1 位或 3 位之后全是 0，太粗）。
2. 判断真实层级：某 7 位前缀只对应一个 9 位码 → 该产品最多到 7 位；5 位同理。
3. 合法 9 位码集合 = 真 9 位码（不以 `00` 结尾）+ 7 位层级码（补 `00`）。
4. 用该集合过滤，在 9 位层级**保留 year** 重新聚合。
5. 只保留**既有产出又有投入**的企业。

层级判断只需要产出侧按 product_id 汇总的小表（几千行），所以先扫一遍拿这张小表，再扫第二遍做过滤聚合，避免把整个 lenth15 装进内存。

In [3]:
# ---- Pass 1：只扫产出侧，按 15 位 product_id 汇总（小表，用于层级判断）----
parts = []
for i, ch in enumerate(pd.read_stata(LENTH15, chunksize=20_000_000), 1):
    o = ch[ch['is_output'] == 1]
    parts.append(o.groupby('product_id', as_index=False)['v'].sum())
    print(f'  pass1 chunk {i}')
g = pd.concat(parts, ignore_index=True).groupby('product_id', as_index=False)['v'].sum()
del parts; gc.collect()
g['product_id'] = g['product_id'].astype(str)
print('产出侧 15 位产品数:', len(g))

# ---- 层级判断 ----
def is_high_level(code):
    # 1 位或 3 位之后全为 0 的聚合码，过粗，剔除
    return (code[1:] == '0' * (len(code) - 1)) or (code[3:] == '0' * (len(code) - 3))

g_low = g[~g['product_id'].apply(is_high_level)].copy()
g_low['p5'] = g_low['product_id'].str[:5]
g_low['p7'] = g_low['product_id'].str[:7]
g_low['p9'] = g_low['product_id'].str[:9]

c7 = g_low.groupby('p7')['p9'].nunique(); single7 = set(c7[c7 == 1].index)  # 7 位前缀唯一
c5 = g_low.groupby('p5')['p7'].nunique(); single5 = set(c5[c5 == 1].index)  # 5 位前缀唯一

five_in_seven  = [x for x in single7 if x.endswith('00')]      # 实为 5 位层级
seven_in_seven = [x for x in single7 if not x.endswith('00')]  # 7 位层级
for i in five_in_seven:
    if i[:-2] in single5:
        seven_in_seven.append(i)
seven_in_seven = [i + '00' for i in seven_in_seven]

p9_true = [x for x in g_low['p9'].drop_duplicates() if not x.endswith('00')]  # 真 9 位
PID9 = set(p9_true + seven_in_seven)
print('合法 9 位码数:', len(PID9))

# 与 bianma.dta（2,778 个 9 位码）核对
bm9 = set(pd.read_stata(BIANMA9)['product_id'].astype(str))
print(f'  与 bianma.dta 交集: {len(PID9 & bm9)}  (bianma: {len(bm9)})')
del g, g_low; gc.collect()

  pass1 chunk 1
  pass1 chunk 2
  pass1 chunk 3
  pass1 chunk 4
  pass1 chunk 5
  pass1 chunk 6
  pass1 chunk 7
  pass1 chunk 8
  pass1 chunk 9
  pass1 chunk 10
  pass1 chunk 11
  pass1 chunk 12
  pass1 chunk 13
  pass1 chunk 14
  pass1 chunk 15
  pass1 chunk 16
  pass1 chunk 17
  pass1 chunk 18
  pass1 chunk 19
  pass1 chunk 20
  pass1 chunk 21
  pass1 chunk 22
  pass1 chunk 23
  pass1 chunk 24
  pass1 chunk 25
产出侧 15 位产品数: 4058
合法 9 位码数: 2778
  与 bianma.dta 交集: 2778  (bianma: 2778)


24

In [4]:
# ---- Pass 2：过滤 + 截 9 位 + 保留 year 聚合 ----
KEYS = ['firm_id', 'product_id', 'is_output', 'year']
parts = []
for i, ch in enumerate(pd.read_stata(LENTH15, chunksize=20_000_000), 1):
    ch['product_id'] = ch['product_id'].astype(str).str[:9]
    ch = ch[ch['product_id'].isin(PID9)]
    parts.append(ch.groupby(KEYS, as_index=False)['v'].sum())
    print(f'  pass2 chunk {i}')
df9 = pd.concat(parts, ignore_index=True).groupby(KEYS, as_index=False)['v'].sum()
del parts; gc.collect()

# ---- 只留既有产出又有投入的企业 ----
f_out = set(df9.loc[df9['is_output'] == 1, 'firm_id'])
f_in  = set(df9.loc[df9['is_output'] == 0, 'firm_id'])
firms = f_out & f_in
df9 = df9[df9['firm_id'].isin(firms)]
df9['year'] = df9['year'].astype(int)

df9.to_stata(OUT_LENTH9, write_index=False)
print('lenth9_year 行数:', f'{len(df9):,}',
      '| 企业:', df9['firm_id'].nunique(),
      '| 产品:', df9['product_id'].nunique())
print('  按年行数:'); print(df9.groupby('year').size().to_string())
del f_out, f_in, firms; gc.collect()

  pass2 chunk 1
  pass2 chunk 2
  pass2 chunk 3
  pass2 chunk 4
  pass2 chunk 5
  pass2 chunk 6
  pass2 chunk 7
  pass2 chunk 8
  pass2 chunk 9
  pass2 chunk 10
  pass2 chunk 11
  pass2 chunk 12
  pass2 chunk 13
  pass2 chunk 14
  pass2 chunk 15
  pass2 chunk 16
  pass2 chunk 17
  pass2 chunk 18
  pass2 chunk 19
  pass2 chunk 20
  pass2 chunk 21
  pass2 chunk 22
  pass2 chunk 23
  pass2 chunk 24
  pass2 chunk 25
lenth9_year 行数: 465,491,679 | 企业: 7191900 | 产品: 2778
  按年行数:
year
2017    218468682
2018    247022997


21

## Step 2　firm×product×year 聚合，外包额 = min(投入, 产出)

以**产出侧为主表** left-merge 投入侧（只保留有产出记录的 firm×product×year）：

- `outsourcing_value = min(total_input, total_output)`
- `production_value  = total_output − outsourcing_value`
- `outsourcing_percen = outsourcing_value / total_output`

In [5]:
out = (df9[df9['is_output'] == 1].drop(columns='is_output')
         .rename(columns={'v': 'total_output'}))
inp = (df9[df9['is_output'] == 0].drop(columns='is_output')
         .rename(columns={'v': 'total_input'}))
del df9; gc.collect()

fpy = out.merge(inp, on=['year', 'firm_id', 'product_id'], how='left')
fpy['total_input'] = fpy['total_input'].fillna(0)
del out, inp; gc.collect()

fpy['outsourcing_value']  = fpy[['total_input', 'total_output']].min(axis=1)
fpy['production_value']   = fpy['total_output'] - fpy['outsourcing_value']
fpy['outsourcing_percen'] = (fpy['outsourcing_value'] / fpy['total_output']).fillna(0)

fpy = fpy[['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value',
           'production_value', 'outsourcing_percen']]
fpy.to_stata(OUT_FPY, write_index=False)
print('firm_product_year_level 行数:', f'{len(fpy):,}',
      '| firm-year:', fpy.groupby(['firm_id', 'year']).ngroups)
print(fpy.groupby('year').agg(rows=('total_output', 'size'),
                              firms=('firm_id', 'nunique')).to_string())

firm_product_year_level 行数: 90,297,401 | firm-year: 12339578
          rows    firms
year                   
2017  40885627  5719322
2018  49411774  6620256


## Step 3　产品级特征 → `product_characteristics.dta`

按 `product_id` 跨 firm、跨年聚合。`num_firms` / `num_firms_outsourcing` **跨年去重**。

这 10 列稍后以 `_p` 后缀并入 `full_data`，用作回归的市场规模控制变量。

In [6]:
num_firms = (fpy[['product_id', 'firm_id']].drop_duplicates()
             .groupby('product_id', as_index=False).size()
             .rename(columns={'size': 'num_firms'}))

num_firms_os = (fpy[fpy['outsourcing_value'] > 0][['product_id', 'firm_id']]
                .drop_duplicates()
                .groupby('product_id', as_index=False).size()
                .rename(columns={'size': 'num_firms_outsourcing'}))

pchars = fpy.groupby('product_id', as_index=False).agg(
    total_output      = ('total_output',      'sum'),
    total_outsourcing = ('outsourcing_value', 'sum'),
    total_production  = ('production_value',  'sum'),
    num_years         = ('year',              'nunique'),
)
pchars = pchars.merge(num_firms,    on='product_id', how='left')
pchars = pchars.merge(num_firms_os, on='product_id', how='left')
pchars['num_firms_outsourcing'] = pchars['num_firms_outsourcing'].fillna(0).astype(int)

pchars['outsourcing_intensity'] = (pchars['total_outsourcing'] / pchars['total_output']).fillna(0)
pchars['avg_output_per_firm']   = pchars['total_output'] / pchars['num_firms']
pchars['avg_output_per_year']   = pchars['total_output'] / pchars['num_years']
pchars['pct_firms_outsourcing'] = (pchars['num_firms_outsourcing'] / pchars['num_firms']).fillna(0)

pchars = pchars.sort_values('num_firms', ascending=False).reset_index(drop=True)
pchars.to_stata(OUT_PCHARS, write_index=False)
print('product_characteristics:', f'{len(pchars):,}', '产品 x', pchars.shape[1], '列')
print('产品级 outsourcing_intensity 均值:', round(pchars['outsourcing_intensity'].mean(), 4))
del num_firms, num_firms_os; gc.collect()

product_characteristics: 2,778 产品 x 11 列
产品级 outsourcing_intensity 均值: 0.151


21

## Step 4　firm×year 汇总：外包强度、中介/外包标记

- `outsourcing_intensity = firm_total_outsource / firm_total_output`
- `is_intermediary = 强度 > 0.90`，`is_outsourcing = 强度 ≥ 0.01`

In [7]:
firm_summary = fpy.groupby(['year', 'firm_id'], as_index=False).agg(
    firm_total_output    = ('total_output',      'sum'),
    firm_total_outsource = ('outsourcing_value', 'sum'),
    n_products           = ('product_id',        'count'),
)
firm_summary['outsourcing_intensity'] = (
    firm_summary['firm_total_outsource'] / firm_summary['firm_total_output']).fillna(0)
firm_summary['is_intermediary'] = (firm_summary['outsourcing_intensity'] > 0.90).astype(int)
firm_summary['is_outsourcing']  = (firm_summary['outsourcing_intensity'] >= 0.01).astype(int)

print('firm-year 观测:', f"{len(firm_summary):,}")
print('中介占比:     {:.2%}'.format(firm_summary['is_intermediary'].mean()))
print('外包企业占比: {:.2%}'.format(firm_summary['is_outsourcing'].mean()))
print('\n按年 firm 数:')
print(firm_summary.groupby('year').size().to_string())

firm-year 观测: 12,339,578
中介占比:     6.24%
外包企业占比: 55.56%

按年 firm 数:
year
2017    5719322
2018    6620256


## Step 5　主产品 = 自产产值 `production_value` 最大

firm×year 内取 `production_value` 最大者（并列取 `product_id` 最小，保证确定性）。同时构造：

- `sales_percen             = total_output / firm_total_output`
- `production_relative_main = production_value / 主产品的 production_value`（**恒 ≤ 1**：主产品按 `production_value` 取最大，分子分母同口径）

In [8]:
fpy_sorted = fpy.sort_values(['year', 'firm_id', 'production_value', 'product_id'],
                             ascending=[True, True, False, True])
main = (fpy_sorted.groupby(['year', 'firm_id'], as_index=False)
        .first()[['year', 'firm_id', 'product_id', 'production_value']]
        .rename(columns={'product_id': 'main_product', 'production_value': 'main_product_production'}))
del fpy_sorted; gc.collect()

df = fpy.merge(main, on=['year', 'firm_id'], how='left')
df['is_main'] = (df['product_id'] == df['main_product']).astype(int)

df = df.merge(firm_summary, on=['year', 'firm_id'], how='left')
df['sales_percen']             = df['total_output'] / df['firm_total_output']
df['production_relative_main'] = df['production_value'] / df['main_product_production']
del main; gc.collect()
print('主产品并入完成，df 行数:', f'{len(df):,}')

主产品并入完成，df 行数: 90,297,401


## Step 6　合并 similarity + 产品特征 → `full_data.dta`（30 列）

1. **similarity**：`full_product_similarity.dta` 是产品对 (product_1, product_2) 的 `input_similarity` / `output_similarity`。对称化后按 (product_id, main_product) 合并；**主产品与自身的相似度定义为 1**。
2. **产品特征**：merge Step 3 的 `product_characteristics`，与 firm-product 级同名的 4 列加 `_p` 后缀。

最终 **30 列** = 20 列基础 + 10 列产品级特征。

In [10]:
# ---- 6a. similarity ----
sim = pd.read_stata(SIM)
sim1 = sim.rename(columns={'product_1': 'product_id', 'product_2': 'main_product'})
sim2 = sim.rename(columns={'product_2': 'product_id', 'product_1': 'main_product'})
sim_lu = (pd.concat([sim1, sim2], ignore_index=True)
            .drop_duplicates(subset=['product_id', 'main_product']))
del sim, sim1, sim2; gc.collect()

for c in ['product_id', 'main_product']:
    df[c]     = df[c].astype(str).str.strip()
    sim_lu[c] = sim_lu[c].astype(str).str.strip()

df = df.merge(sim_lu[['product_id', 'main_product', 'input_similarity', 'output_similarity']],
              on=['product_id', 'main_product'], how='left')
df.loc[df['is_main'] == 1, ['input_similarity', 'output_similarity']] = 1.0
del sim_lu; gc.collect()

cols20 = ['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value', 'production_value',
          'outsourcing_percen', 'sales_percen', 'production_relative_main', 'is_main', 'main_product',
          'main_product_production', 'input_similarity', 'output_similarity', 'firm_total_output',
          'firm_total_outsource', 'n_products', 'outsourcing_intensity', 'is_intermediary', 'is_outsourcing']
df = df[cols20]

# ---- 6b. 产品级特征（同名列加 _p 后缀）----
pc = pchars.rename(columns={
    'total_output':          'total_output_p',
    'total_outsourcing':     'total_outsourcing_p',
    'total_production':      'total_production_p',
    'outsourcing_intensity': 'outsourcing_intensity_p',
})
pc['product_id'] = pc['product_id'].astype(str).str.strip()
df = df.merge(pc, on='product_id', how='left')

df = df.sort_values(['year', 'firm_id', 'total_output'],
                    ascending=[True, True, False]).reset_index(drop=True)
df.to_stata(OUT_FULLDATA, write_index=False)
print('full_data.dta:', f'{len(df):,}', '行 x', df.shape[1], '列')
print('  企业:', df['firm_id'].nunique(), '| 产品:', df['product_id'].nunique())
print('  列名:', list(df.columns))

full_data.dta: 90,297,401 行 x 30 列
  企业: 7191900 | 产品: 2778
  列名: ['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value', 'production_value', 'outsourcing_percen', 'sales_percen', 'production_relative_main', 'is_main', 'main_product', 'main_product_production', 'input_similarity', 'output_similarity', 'firm_total_output', 'firm_total_outsource', 'n_products', 'outsourcing_intensity', 'is_intermediary', 'is_outsourcing', 'total_output_p', 'total_outsourcing_p', 'total_production_p', 'num_years', 'num_firms', 'num_firms_outsourcing', 'outsourcing_intensity_p', 'avg_output_per_firm', 'avg_output_per_year', 'pct_firms_outsourcing']


## Step 7　与现有 `full_data.dta` 对比验证

与旧底表逐列比对（chunked 读，省内存）。

**预期不一致**：`is_main` / `main_product_production` / `production_relative_main` / `input_similarity` / `output_similarity`——旧版主产品用 `total_output` 最大，本版改为 `production_value` 最大。

其余列若有差异，说明清洗链条与旧版不同，需要排查。

In [11]:
if not OLD_FULLDATA.exists():
    print('现有 full_data.dta 不存在，跳过对比')
else:
    num_cols = df.select_dtypes('number').columns.tolist()
    n_old, old_cols, sums_old = 0, None, None
    for ch in pd.read_stata(OLD_FULLDATA, chunksize=5_000_000):
        n_old += len(ch)
        if old_cols is None:
            old_cols = list(ch.columns)
        common = [c for c in num_cols if c in ch.columns]
        s = ch[common].sum()
        sums_old = s if sums_old is None else sums_old + s

    print(f'行数:  旧 {n_old:,}   新 {len(df):,}   match = {n_old == len(df)}')
    print(f'列数:  旧 {len(old_cols)}   新 {df.shape[1]}   列集相同 = {set(old_cols) == set(df.columns)}')
    print(f'仅旧有: {sorted(set(old_cols) - set(df.columns))}')
    print(f'仅新有: {sorted(set(df.columns) - set(old_cols))}')

    cmp = pd.DataFrame({'old': sums_old, 'new': df[sums_old.index].sum()})
    cmp['rel_diff'] = ((cmp['new'] - cmp['old']) / cmp['old'].abs().replace(0, np.nan)).abs()

    EXPECTED_DIFF = ['is_main', 'main_product_production', 'production_relative_main',
                     'input_similarity', 'output_similarity']
    cmp['note'] = np.where(cmp.index.isin(EXPECTED_DIFF), '预期不同(主产品口径)', '')

    chk = cmp[~cmp.index.isin(EXPECTED_DIFF)]
    bad = chk[chk['rel_diff'] > 1e-9]
    print('\n【应当一致的列】rel_diff > 1e-9 即异常:')
    print(('  异常:\n' + bad.to_string()) if len(bad) else '  全部一致 OK')

    print('\n【全列对比】')
    print(cmp.round(8).to_string())

现有 full_data.dta 不存在，跳过对比


## Step 8　描述统计

In [12]:
# 8.1 企业分类（firm-year）
fy = firm_summary.copy()
fy['ftype'] = np.where(fy['is_intermediary'] == 1, 'Intermediary',
               np.where(fy['outsourcing_intensity'] >= 0.01, 'Outsourcing', 'Pure self'))

tab = fy.groupby('ftype').agg(
    firm_years   = ('firm_id', 'size'),
    unique_firms = ('firm_id', 'nunique'),
    total_output = ('firm_total_output', 'sum'),
).reset_index()
tab['pct_firm_years'] = tab['firm_years'] / tab['firm_years'].sum()
print('=== 8.1 企业分类（firm-year）===')
print(tab.to_string(index=False))

# 8.2 Scope gap：产品对分解（剔除中介）
non_int = df[df['is_intermediary'] != 1]
bk = np.where(non_int['outsourcing_percen'] <= 0, 'Pure self',
      np.where(non_int['outsourcing_percen'] >= 1, 'Pure outsourcing', 'Mixed'))
sg = non_int.groupby(bk).agg(n_pairs=('total_output', 'size'), sales=('total_output', 'sum'))
sg['pct_pairs'] = sg['n_pairs'] / sg['n_pairs'].sum()
sg['pct_sales'] = sg['sales']   / sg['sales'].sum()
print('\n=== 8.2 Scope gap（剔除中介）===')
print(sg[['n_pairs', 'pct_pairs', 'pct_sales']].round(4).to_string())

# 8.3 外包普遍率与强度分布
print('\n=== 8.3 外包普遍率（按年）===')
print(firm_summary.groupby('year')['is_outsourcing'].mean().round(4).to_string())

os_firms = firm_summary[(firm_summary['is_intermediary'] == 0) &
                        (firm_summary['outsourcing_intensity'] > 0)]
print('\n外包强度分布（剔除中介、强度>0）:')
print(os_firms['outsourcing_intensity'].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(4).to_string())

=== 8.1 企业分类（firm-year）===
       ftype  firm_years  unique_firms  total_output  pct_firm_years
Intermediary      769618        655685  4.866027e+13        0.062370
 Outsourcing     6086847       4044098  2.952615e+14        0.493278
   Pure self     5483113       3924179  7.533535e+13        0.444352

=== 8.2 Scope gap（剔除中介）===
                   n_pairs  pct_pairs  pct_sales
Mixed             17851990     0.2042     0.7299
Pure outsourcing  15322288     0.1752     0.0713
Pure self         54258839     0.6206     0.1988

=== 8.3 外包普遍率（按年）===
year
2017    0.5326
2018    0.5755

外包强度分布（剔除中介、强度>0）:
count    6.947338e+06
mean     2.612000e-01
std      2.576000e-01
min      0.000000e+00
10%      6.800000e-03
25%      3.800000e-02
50%      1.686000e-01
75%      4.379000e-01
90%      6.847000e-01
99%      8.763000e-01
max      9.000000e-01


## 输出文件

全部落在 `Empirical1_data/`（原始数据 `Data/` 不动）：

| 文件 | 内容 |
|---|---|
| `lenth15_year.dta` | 01_cleaning.do 产物：firm×product(15位)×is_output×**year** |
| `lenth9_year.dta` | 9 位码标准化 + firm 交集后，含 **year** |
| `firm_product_year_level.dta` | firm×product×year，7 列 |
| `product_characteristics.dta` | 产品级特征，11 列 |
| `full_data.dta` | **第一阶段回归底表，30 列** |
| `1718_cleaned19_year.dta` | 19 位清洗中间结果（可删） |
| `_codetable19.dta` | 编码表副本（可删） |

**口径**：外包额 = min(投入,产出)；主产品 = **production_value 最大**；中介 = 强度 > 0.90；外包企业 = 强度 ≥ 0.01。**不做**对角线对冲与 dominant 处理（那是 IO 表专用）。